# Set 3 오답노트

## 검토한 파일

- `set_01_06_answer/03_question.ipynb`
- `set_01_06_answer/03_answer.ipynb`
- `set_01_06_answer/set_03_answer.ipynb`
- `set_01_06_answer/set_03_cbi.ipynb`
- `00_trying/01/03_question.ipynb`
- `00_trying/02/03_question.ipynb`

## 최종 답

- Q01: 성능지표 평균 **11.01**
- Q02: 가장 큰 상관계수 **0.95** (`num_of_ratings`)
- Q03: 가장 우수한 이웃 개수 **k = 3**


## Q01 — 상위 이상치와 성능지표

### 주석 질문: 절댓값을 사용해야 하는가?

일반적으로 평균에서 양쪽으로 멀리 떨어진 이상치를 찾는다면 다음처럼 절댓값을 사용한다.

```python
abs(sales - sales_mean) > 2 * sales_std
```

하지만 이 문제는 판매지수가 특별히 **큰 제품**을 주목받는 제품으로 정의하고, 기준 답안도 상위 이상치만 선택한다. 따라서 문제 의도에 맞는 조건은 다음과 같다.

```python
sales > sales_mean + 2 * sales_std
```

또는 z-score 형태로 같은 조건을 표현할 수 있다.

```python
(sales - sales_mean) / sales_std > 2
```

두 식은 다음과 같이 동치다.

```text
(sales - mean) / std > 2
sales - mean > 2 × std
sales > mean + 2 × std
```

### 절댓값을 사용해도 이번 결과가 같았던 이유

실제 데이터의 값은 다음과 같다.

```text
sales 평균       = 29.7523
sales 표준편차   = 58.3996
상한선           = 29.7523 + 2 × 58.3996 ≈ 146.5515
하한선           = 29.7523 - 2 × 58.3996 ≈ -87.0469
```

판매지수에는 `-87.05`보다 작은 값이 없기 때문에 절댓값 조건을 사용해도 낮은 쪽 이상치가 추가되지 않는다. 두 방법 모두 16개 제품과 답 `11.01`이 나왔지만, 다른 데이터에서는 결과가 달라질 수 있다. 시험에서는 문제의 '큰 값'이라는 표현에 맞춰 상한 조건을 사용한다.

### 권장 풀이

```python
df_q1 = df.copy()
sales_mean = df_q1['sales'].mean()
sales_std = df_q1['sales'].std()
upper_limit = sales_mean + 2 * sales_std

focus = df_q1.loc[df_q1['sales'] > upper_limit].copy()
performance = (
    focus['ROM'] / 32
    + focus['RAM'] / 2
    + focus['num_rear_camera']
    + focus['num_front_camera']
    + focus['battery_capacity'] / 1000
)
answer_q1 = round(performance.mean(), 2)
display(answer_q1)  # 11.01
```

긴 덧셈을 줄바꿈할 때는 전체 식을 괄호로 감싸야 한다. `loc`로 필터링한 DataFrame에 새 열을 추가하려면 `.copy()`를 사용해 `SettingWithCopyWarning`을 예방한다. pandas의 `Series.std()`는 기본적으로 표본 표준편차(`ddof=1`)를 사용하며 기준 답안도 같은 방식을 사용한다.


## Q02 — 지정 변수와 sales의 상관관계

### 지정된 변수만 선택해야 하는 이유

`corr()`을 전체 DataFrame에 적용한 뒤 `['sales']`를 선택해도 sales와 모든 숫자형 변수의 상관계수가 계산된다. 그러나 문제는 다음 5개 변수만 비교 대상으로 지정했다.

```python
corr_cols = [
    'battery_capacity', 'ratings', 'num_of_ratings',
    'sales_price', 'discount_percent', 'sales'
]
```

전체 숫자형 열을 사용하면 `ROM`, `RAM`, 카메라 개수처럼 문제에서 요구하지 않은 변수가 최댓값으로 선택될 가능성이 있다. 현재 데이터에서는 답이 우연히 같지만, 문제 범위를 명확히 지키고 실수를 방지하려면 지정 열만 선택한 뒤 `corr()`을 계산한다.

### 필터링 조건

다음 두 표현은 같다.

```python
~(df['num_rear_camera'] == 1)
df['num_rear_camera'] != 1
```

두 번째 표현이 문제 문장인 '1개인 제품은 제외'와 직접 대응해 읽기 쉽다. 필터링 후 인덱스를 다시 만들 필요는 없으므로 `reset_index()`는 생략할 수 있다.

### 절댓값으로 변수를 선택하고 원래 부호 확인하기

실제 상관계수는 다음과 같다.

```text
battery_capacity     0.025680
ratings              0.226075
num_of_ratings       0.949114
sales_price         -0.247760
discount_percent     0.223471
```

상관계수 절댓값이 가장 큰 변수는 `num_of_ratings`이고 원래 상관계수도 양수이므로 결과는 `0.95`다. 강한 음의 상관관계가 존재할 수도 있으므로 단순히 원래 값에 `.max()`를 적용하면 안 된다. `pearsonr()` 결과 다섯 개에 바로 `max()`를 적용하는 방식도 같은 이유로 안전하지 않다.

```python
df_q2 = df.loc[df['num_rear_camera'] != 1, corr_cols].copy()
corr_sales = df_q2.corr(method='pearson')['sales'].drop(index='sales')

best_col = corr_sales.abs().idxmax()
best_corr = corr_sales.loc[best_col]

display(best_col)             # num_of_ratings
display(round(best_corr, 2))  # 0.95
```

문제가 절댓값 자체를 답으로 요구한다고 해석하면 `round(corr_sales.abs().max(), 2)`를 사용해도 된다. 이 데이터의 최댓값은 양수라 두 방식의 답이 같다.


## Q03 — One Hot Encoding과 k-NN 회귀

### 1. 종속변수를 먼저 분리하고 독립변수만 인코딩

전체 DataFrame에 `get_dummies()`를 적용한 뒤 sales를 제거해도 이 데이터에서는 작동한다. 하지만 전처리 대상은 독립변수이므로 X와 y를 먼저 나누는 흐름이 명확하다.

```python
X = df.drop(columns='sales').copy()
y = df['sales'].copy()

X_ohe = pd.get_dummies(
    X,
    columns=['screen_size'],
    drop_first=False
)
display(X_ohe.shape)  # (430, 14)
```

`pd.get_dummies()`의 `columns`에는 열 하나만 지정하더라도 `['screen_size']`처럼 리스트를 전달한다. 원래 독립변수는 10개이고 `screen_size` 1개가 5개의 더미변수로 교체되므로 `10 - 1 + 5 = 14개`가 된다. 문제에서 제시한 14개와 일치하는지 반드시 확인한다.

`pd.get_dummies(X)`처럼 columns를 생략하면 모든 object 열을 자동 인코딩하므로 현재 데이터에서는 동일하게 동작한다. 시험에서는 어떤 열을 인코딩했는지 명확하게 드러내기 위해 `columns=['screen_size']`를 쓰는 편이 좋다.

열 이름의 공백을 밑줄로 바꿀 때는 결과를 다시 대입해야 한다. 모델 학습에 필수적인 작업은 아니지만 이후 열 이름을 직접 참조할 때 편하다.

```python
X_ohe.columns = X_ohe.columns.str.replace(' ', '_')
```

### 2. 분할 후 학습 데이터로만 정규화

```python
X_train, X_test, y_train, y_test = train_test_split(
    X_ohe, y, test_size=0.2, random_state=123
)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
```

평가 데이터는 학습 데이터에서 구한 최솟값과 최댓값으로 변환해야 한다. 따라서 학습 데이터에는 `fit_transform()`, 평가 데이터에는 `transform()`만 사용한다. 종속변수 y까지 함께 정규화하면 RMSE의 단위가 원래 sales 단위가 아니게 된다. 이 문제에서는 독립변수만 정규화한다.

### 3. 주석 질문: k-NN에는 왜 `random_state`가 없는가?

`KNeighborsRegressor`는 학습 과정에서 임의의 초기값이나 무작위 표본 추출을 사용하지 않는다. 입력 데이터와 k가 같으면 가까운 이웃과 예측값이 결정적으로 정해지므로 `random_state` 매개변수가 없다. 이 문제의 무작위성은 데이터 분할에서만 발생하므로 `train_test_split(..., random_state=123)`에 seed를 지정한다.

`KNeighborsRegressor(k)`도 동작하지만 `KNeighborsRegressor(n_neighbors=k)`라고 쓰면 숫자의 의미가 명확하다.

### 4. RMSE는 가장 작은 값을 선택

```python
rmse_by_k = {}

for k in [3, 5, 7, 9, 11]:
    model = KNeighborsRegressor(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    rmse_by_k[k] = mean_squared_error(y_test, y_pred) ** 0.5

rmse = pd.Series(rmse_by_k, name='RMSE')
best_k = rmse.idxmin()
```

실제 결과:

```text
k=3     RMSE=40.440549
k=5     RMSE=48.800827
k=7     RMSE=53.186755
k=9     RMSE=55.484384
k=11    RMSE=56.160703
```

RMSE는 오차이므로 클수록 좋은 점수가 아니라 **작을수록 좋은 점수**다. 최솟값 자체는 `rmse.min()`, 그 최솟값의 인덱스인 k는 `rmse.idxmin()`으로 구한다. 따라서 최적의 이웃 개수는 `3`이다.


## 핵심 암기

1. 일반적인 양쪽 이상치는 `abs(x - mean) > 2×std`, 상위 이상치는 `x > mean + 2×std`다.
2. 문제에서 비교 변수를 지정하면 해당 열만 선택한 뒤 분석한다.
3. 절댓값이 가장 큰 상관관계는 `.abs().idxmax()`로 변수명을 찾고 원본 Series에서 부호를 확인한다.
4. One Hot Encoding은 X에 적용하고 생성된 독립변수 개수를 문제 조건과 대조한다.
5. `get_dummies(columns=...)`의 columns에는 리스트를 전달한다.
6. scaler는 학습 데이터로만 `fit`하고 평가 데이터에는 `transform`만 적용한다.
7. k-NN은 결정적 알고리즘이므로 모델 자체에 `random_state`가 없다.
8. RMSE는 작을수록 좋으며 최적 k는 `idxmin()`으로 찾는다.
